In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

In [ ]:
movies = pd.read_csv('movies_metadata.csv', low_memory = False)
keywords = pd.read_csv('keywords.csv')
credits = pd.read_csv('credits.csv')

In [ ]:
print(movies.head())

In [ ]:
print(keywords.head())

In [ ]:
print(credits.head())

In [ ]:
print(movies.info())

In [ ]:
print(keywords.info())

In [ ]:
print(credits.info())

In [ ]:
movies.dtypes

In [ ]:
movies['id'] = movies['id'].astype(str)
movies = movies[movies['id'].str.isnumeric()]
movies['id'] = movies['id'].astype(int)
keywords['id'] = keywords['id'].astype(int)
credits['id'] = credits['id'].astype(int)
df = movies.merge(keywords, on = 'id', how = 'left').merge(credits , on = 'id', how = 'left')

# Data Cleaning

In [ ]:
df.head()

In [ ]:
print(df.info())

In [ ]:
def parse_names(text):
    try:
        items = ast.literal_eval(text)
        return [item['name'] for item in items]
    except (ValueError, SyntaxError, TypeError):
        pass
    return np.nan

df['genre_list'] = df['genres'].apply(parse_names)
df['keywords_list'] = df['keywords'].apply(parse_names)
df['cast_list'] = df['cast'].apply(parse_names)

In [ ]:
def get_director (text):
    try:
        crew = ast.literal_eval(text)
        for person in crew:
            if person.get('job') == 'Director':
                return person['name']
    except (ValueError, SyntaxError, TypeError):
        pass
    return np.nan

df['director'] = df['crew'].apply(get_director)

In [ ]:
cols_to_keep = ['id', 'title', 'release_date','budget', 'revenue', 'runtime', 'vote_average', 'vote_count',
                 'popularity', 'genre_list', 'keywords_list', 'cast_list', 'director']
films = df[cols_to_keep]

In [ ]:
print(films.head())

In [ ]:
print(films.info())

In [ ]:
films['release_date'] = pd.to_datetime(films['release_date'], errors = 'coerce')
films['budget'] = pd.to_numeric(films['budget'], errors = 'coerce')
films['popularity'] = pd.to_numeric(films['popularity'], errors = 'coerce')

In [ ]:
print(films.isnull().sum())

In [ ]:
films['budget'] = films['budget'].replace(0, np.nan)
films['revenue'] = films['revenue'].replace(0, np.nan)
films['runtime'] = films['runtime'].replace(0, np.nan)

In [ ]:
films = films.dropna()

In [ ]:
films['release_year'] = films['release_date'].dt.year
films['release_month'] = films['release_date'].dt.month

In [ ]:
print(films.describe())

In [ ]:
print(films.duplicated(subset = 'id').sum())

In [ ]:
films = films.drop_duplicates(subset = 'id')

## Column Budget

In [ ]:
plt.figure(figsize=(8,6))

plt.boxplot(films['budget'])

plt.show()

In [ ]:
budgetQ3 = films['budget'].quantile(0.75)
print(films[films['budget'] > 1.5 * budgetQ3]['budget'].count())

In [ ]:
films = films[~((films['release_year'] < 1991) & (films['budget'] > 100000000))]

## Cloumn Revenue

In [ ]:
plt.figure(figsize=(8,6))

plt.boxplot(films['revenue'])

plt.show()

In [ ]:
budgetQ3 = films['revenue'].quantile(0.75)
print(films[films['revenue'] > 1.5 * budgetQ3]['revenue'].count())